In [1]:
import os,sys,subprocess,glob,cftime,importlib,pickle,itertools
from datetime import datetime
import xarray as xr
import numpy as np
import pandas as pd
import scipy

sys.path.append('../REA_with_CESM2')
from ensembles.ensemble_GKLT import ensemble_GKLT,get_weight_for_selection

sys.path.append('../REA_heat_wEU_JJA')
from experiment_configuration.experiment import experiment

def import_from(module, name):
    module = __import__(module, fromlist=[name])
    return getattr(module, name)

import warnings
warnings.filterwarnings('ignore')

from sup.sup_plotting import *
from sup.merged_ensembles import *

%load_ext autoreload
%autoreload 2

## initialization

In [2]:
data_dir = '/work/bb1152/u290372/REA_output/heat_wEU_JJA/NCAR/CESM2/'
climates = ['ssp370-2025', 'piControl']

In [3]:
threshold = 2.2122549116517263

In [4]:
with open('pickles/REA_heat_1D_stats.pkl', 'rb') as fl:
    ensembles = pickle.load(fl)

In [4]:
def weighted_expected_value(
        tas, 
        weight, 
        threshold,
        x,
        ):
    return ((tas >= threshold) * x * weight).sum('sim') / ((tas >= threshold) * weight).sum('sim')

In [5]:
def preprocess(nc,var):
    if nc.lon[0] == 0:
        nc = nc.roll(lon=144, roll_coords=True)
    nc = nc.assign_coords(lon=(nc.lon + 180) % 360 - 180).sel(dict(lat=slice(-20,90))).load()
    if var == 'tos':
        if nc['tos'].max() > 273:
            nc['tos'] -= 273.15
            nc['tos'].values[nc['tos'] < -200] = np.nan
    nc[var] = nc[var].mean('time')
    return nc

## REA

In [7]:

for var in ['pr', 'tos', 'mrsos', 'tas', 'eke500', 'zg500','stream500','ua500','va500','ua200']:
    print(var)
    var_name_in_file = var

    for r,climate in enumerate(climates):

        rt = 100
        ens = ensembles['piControl-LE-all']
        threshold = weighted_quantile(ens['data']['tas_anom'].mean('time'), (1 - 1/rt), sample_weight=ens['data']['weight'])
        threshold += ensembles[f'{climate}-LE-all']['data']['tas_absolute'].values.mean()


        l = []
        for i in range(1,6):
            ens_name = f"{climate}-x{i}"
            try:
                with xr.open_mfdataset(f"{data_dir}/{ens_name}/meta/obs/*/*", concat_dim='sim', combine='nested') as nc:
                    tas = nc['obs'].mean('time').load()

                with xr.open_dataset(f"{data_dir}/{ens_name}/meta/weight_season_{ens_name}.nc") as nc:
                    weight = nc['weight'].load()

                with xr.open_mfdataset(f"{data_dir}/{ens_name}/mon/*/{var}/*/*", concat_dim='sim', combine='nested') as nc:
                    tmp = preprocess(nc,var_name_in_file)[var_name_in_file].load() 
                    l.append(
                        weighted_expected_value(
                            tas=tas, 
                            weight=weight, 
                            threshold=threshold, 
                            x=tmp
                        )
                    )
                    attrs = nc.attrs
            except:
                print(ens_name, 'fail')
        y = xr.concat(l, dim='sim').mean('sim')
        xr.Dataset({var:y}).to_netcdf(f'average_maps/{var}_{climate}_rea_100.nc')


pr
tos
mrsos
ssp370-2025-x1 fail
ssp370-2025-x2 fail
piControl-x1 fail
piControl-x2 fail
tas
eke500
ssp370-2025-x1 fail
piControl-x1 fail
zg500
stream500
ssp370-2025-x1 fail
piControl-x1 fail
ua500
ssp370-2025-x1 fail
piControl-x1 fail
va500
ssp370-2025-x1 fail
piControl-x1 fail
ua200
ssp370-2025-x1 fail
piControl-x1 fail


## REA dry

In [8]:
climate = 'ssp370-2025'
rt = 100
ens = ensembles['piControl-LE-all']
threshold = weighted_quantile(ens['data']['tas_anom'].mean('time'), (1 - 1/rt), sample_weight=ens['data']['weight'])
threshold += ensembles[f'{climate}-LE-all']['data']['tas_absolute'].values.mean()

for var in ['tas', 'zg500']:
    print(var)
    var_name_in_file = var
    l = []
    for ens_name in [
        "ssp370-2025-dry-x6",
        "ssp370-2025-dry-x9",
        "ssp370-2025-dry-x11",
        "ssp370-2025-wet-x7",
        "ssp370-2025-wet-x8",
        "ssp370-2025-wet-x12",
    ]:
        try:
            with xr.open_mfdataset(f"{data_dir}/{ens_name}/meta/obs/*/*", concat_dim='sim', combine='nested') as nc:
                tas = nc['obs'].mean('time').load()

            with xr.open_dataset(f"{data_dir}/{ens_name}/meta/weight_season_{ens_name}.nc") as nc:
                weight = nc['weight'].load()

            with xr.open_mfdataset(f"{data_dir}/{ens_name}/mon/*/{var}/*/*", concat_dim='sim', combine='nested') as nc:
                tmp = preprocess(nc,var_name_in_file)[var_name_in_file].load() 
                l.append(
                    weighted_expected_value(
                        tas=tas, 
                        weight=weight, 
                        threshold=threshold, 
                        x=tmp
                    )
                )
                attrs = nc.attrs
        except:
            print(ens_name, 'fail')
    y = xr.concat(l, dim='sim').mean('sim')
    xr.Dataset({var:y}).to_netcdf(f'average_maps/{var}_ssp370-2025-dry_rea_100.nc')


tas
zg500


## SST STD map for initial

In [7]:
with xr.open_mfdataset(f"{data_dir}/piControl-initial/mon/*/tos/*/*", concat_dim='sim', combine='nested') as nc:
    tmp = preprocess(nc,'tos')['tos'].load() 

In [ ]:
xr.Dataset({'tos':tmp.std('sim')}).to_netcdf(f'std_maps/tos_piControl-initial_std.nc')